# PSYLIQ Project 1 — Scam Execution & Victim Profiling
**Phase 2: Python + SQL — Validation, Analysis, Visualization**

Data source: `Data_Matrix_P1.xlsx` (Phase 1 output)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from collections import Counter
import re

pd.set_option('display.width', 140)


## Setup — Load All Sheets

In [2]:
xl = pd.ExcelFile('Data_Matrix_P1.xlsx')

annual = xl.parse('Annual_Summary')
age = xl.parse('Age_Group')
geo = xl.parse('Geographic_Report')
crime_types = xl.parse('Scam_Crime_Types')
payment = xl.parse('Payment_Methods')
gov = xl.parse('Government_Actions')
tactics = xl.parse('Scam_Tactics')
script = xl.parse('Scam_Script')

print("Loaded sheets:", {n: d.shape for n, d in
      [('annual', annual), ('age', age), ('geo', geo), ('crime_types', crime_types),
       ('payment', payment), ('gov', gov), ('tactics', tactics), ('script', script)]})


Loaded sheets: {'annual': (5, 3), 'age': (30, 4), 'geo': (285, 5), 'crime_types': (142, 4), 'payment': (51, 4), 'gov': (10, 11), 'tactics': (44, 8), 'script': (35, 5)}


## Data Validation & Cleaning

In [3]:
print("\n--- VALIDATION ---")
for name, df in [('age', age), ('geo', geo), ('crime_types', crime_types),
                  ('payment', payment), ('gov', gov), ('tactics', tactics)]:
    print(f"{name}: duplicates={df.duplicated().sum()}, nulls={df.isnull().sum().sum()}")

# Clean Government_Actions mixed-type columns
gov_clean = gov.copy()

def parse_date_safe(val):
    if isinstance(val, str):
        if val == 'Not specified in report':
            return pd.NaT
        if re.match(r'^\d{4}-\d{2}$', val):
            return pd.to_datetime(val + '-01')
    return pd.to_datetime(val, errors='coerce')

gov_clean['Date_of_Action_Clean'] = gov_clean['Date_of_Action'].apply(parse_date_safe)
gov_clean['Date_Confidence'] = gov_clean['Date_of_Action'].apply(
    lambda v: 'Missing' if v == 'Not specified in report'
    else ('Month-only' if isinstance(v, str) and re.match(r'^\d{4}-\d{2}$', v) else 'Exact'))

gov_clean['Est_Funds_Recovered_Clean'] = pd.to_numeric(
    gov_clean['Est_Funds_Recovered_USD'], errors='coerce')

print("\nGovernment_Actions after cleaning:")
print(gov_clean[['Action_ID', 'Date_of_Action', 'Date_of_Action_Clean',
                  'Date_Confidence', 'Est_Funds_Recovered_USD', 'Est_Funds_Recovered_Clean']])

n_missing_funds = gov_clean['Est_Funds_Recovered_Clean'].isnull().sum()
print(f"\nRows with unusable recovery figure: {n_missing_funds} (excluded from AVG calc below)")



--- VALIDATION ---
age: duplicates=0, nulls=0
geo: duplicates=0, nulls=0
crime_types: duplicates=0, nulls=0
payment: duplicates=0, nulls=0
gov: duplicates=0, nulls=0
tactics: duplicates=0, nulls=2

Government_Actions after cleaning:
  Action_ID           Date_of_Action Date_of_Action_Clean Date_Confidence  Est_Funds_Recovered_USD  Est_Funds_Recovered_Clean
0   P1-A001      2020-03-02 00:00:00           2020-03-02           Exact                 11000000               1.100000e+07
1   P1-A002      2020-02-19 00:00:00           2020-02-19           Exact                 15466278               1.546628e+07
2   P1-A003  Not specified in report                  NaT         Missing  Not specified in report                        NaN
3   P1-A004      2021-01-27 00:00:00           2021-01-27           Exact                127500000               1.275000e+08
4   P1-A005               2022-05-00                  NaT           Exact                  8400000               8.400000e+06
5   P1-A00

## Analysis — Module 1.1: Victim Demographics

In [4]:
print("\n--- 1.1 VICTIM DEMOGRAPHICS ---")

age_trend = age[age['Age_Group'].isin(['20-29', '60+'])][['Year', 'Age_Group', 'Reported_Count']]
age_trend_pivot = age_trend.pivot(index='Year', columns='Age_Group', values='Reported_Count')
print("\nAge trend (20-29 vs 60+):")
print(age_trend_pivot)

growth_2029 = (age_trend_pivot['20-29'].iloc[-1] - age_trend_pivot['20-29'].iloc[0]) / age_trend_pivot['20-29'].iloc[0] * 100
growth_60plus = (age_trend_pivot['60+'].iloc[-1] - age_trend_pivot['60+'].iloc[0]) / age_trend_pivot['60+'].iloc[0] * 100
print(f"\n20-29 growth 2020->2024: {growth_2029:.1f}%")
print(f"60+ growth 2020->2024: {growth_60plus:.1f}%")

top_states_2024 = geo[geo['Year'] == 2024].sort_values('Reported_Complaints', ascending=False).head(10)
print("\nTop 10 states, 2024:")
print(top_states_2024[['Rank', 'State', 'Reported_Complaints', 'Reported_Loss_USD']])



--- 1.1 VICTIM DEMOGRAPHICS ---

Age trend (20-29 vs 60+):
Age_Group  20-29     60+
Year                    
2020       70791  105301
2021       69390   92371
2022       57978   88262
2023       62410  101068
2024       71399  147127

20-29 growth 2020->2024: 0.9%
60+ growth 2020->2024: 39.7%

Top 10 states, 2024:
     Rank           State  Reported_Complaints  Reported_Loss_USD
228     1      California                96265         2539041635
229     2           Texas                62347         1351598183
230     3         Florida                52191         1071909632
231     4        New York                36468          903975003
232     5    Pennsylvania                27838          400082312
233     6        Illinois                25446          479054271
234     7            Ohio                24915          278038028
235     8         Indiana                23659          125093323
236     9  North Carolina                22021          324287947
237    10         Arizo

In [11]:
us_state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL',
    'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA',
    'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR',
    'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD',
    'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA',
    'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'Puerto Rico': 'PR', 'Guam': 'GU'
}

geo_map_data = geo[geo['Year'] == 2024].copy()
geo_map_data['State_Abbrev'] = geo_map_data['State'].map(us_state_abbrev)

# check for anything that didn't match (territories not in the dict, etc.)
unmatched = geo_map_data[geo_map_data['State_Abbrev'].isnull()]['State'].unique()
print("States/territories with no abbreviation match (will stay gray, safe to ignore):", unmatched)

fig = px.choropleth(
    geo_map_data.dropna(subset=['State_Abbrev']),
    locations='State_Abbrev', locationmode="USA-states",
    color='Reported_Complaints', scope="usa",
    color_continuous_scale="Reds", title="2024 Complaint Volume by State"
)
fig.write_html("geo_heatmap_2024.html")
fig.show()

States/territories with no abbreviation match (will stay gray, safe to ignore): <StringArray>
['U.S. Minor Outlying Islands', 'American Samoa', 'Virgin Islands, U.S.', 'Northern Mariana Islands']
Length: 4, dtype: str


## Analysis — Module 1.2: Scammer Tactics

In [6]:
print("\n--- 1.2 SCAMMER TACTICS ---")

top5_scam_types_2024 = crime_types[crime_types['Year'] == 2024].sort_values(
    'Reported_Complaints', ascending=False).head(5)
print("\nTop 5 scam/crime types, 2024 (by complaint volume):")
print(top5_scam_types_2024[['Scam_Type', 'Reported_Complaints', 'Reported_Loss_USD']])

urgency_counts = Counter(tactics['Urgency_Language'].dropna())
print("\nTop 5 most frequent urgency-language phrases (Scam_Tactics sheet):")
for phrase, count in urgency_counts.most_common(5):
    print(f"  {count}x  {phrase}")

top3_payment_2024 = payment[payment['Year'] == 2024].sort_values(
    'Reported_Count', ascending=False).head(3)
print("\nTop 3 payment methods demanded, 2024:")
print(top3_payment_2024[['Payment_Method', 'Reported_Count', 'Reported_Loss_USD']])

payment_2020 = payment[payment['Year'] == 2020].sort_values('Reported_Count', ascending=False)
payment_2024 = payment[payment['Year'] == 2024].sort_values('Reported_Count', ascending=False)
print("\nPayment method shift, 2020 (top3) vs 2024 (top3):")
print("2020:", list(payment_2020['Payment_Method'].head(3)))
print("2024:", list(payment_2024['Payment_Method'].head(3)))



--- 1.2 SCAMMER TACTICS ---

Top 5 scam/crime types, 2024 (by complaint volume):
                     Scam_Type  Reported_Complaints  Reported_Loss_USD
116          Phishing/Spoofing               193407           70013036
117                  Extortion                86415          143185736
118       Personal Data Breach                64882         1453296303
119  Non-Payment/ Non-Delivery                49572          785436888
120                 Investment                47919         6570639864

Top 5 most frequent urgency-language phrases (Scam_Tactics sheet):
  15x  Not specified in report
  2x  Threat of data destruction/exposure
  1x  "Time-sensitive" stimulus/benefit access framing
  1x  Threats or charades to compel compliance
  1x  "Pay to get early access" / limited availability pressure

Top 3 payment methods demanded, 2024:
            Payment_Method  Reported_Count  Reported_Loss_USD
41            Credit Cards          108881          275000000
42  Payment App or Ser

## Analysis — Module 1.3: Government Response (Real SQL)

In [7]:
print("\n--- 1.3 GOVERNMENT RESPONSE (SQL) ---")

conn = sqlite3.connect(':memory:')
gov_clean.to_sql('actions', conn, if_exists='replace', index=False)

query = """
SELECT Agency,
       COUNT(*) AS num_actions,
       AVG(Est_Funds_Recovered_Clean) AS avg_recovered,
       SUM(Est_Funds_Recovered_Clean) AS total_recovered
FROM actions
WHERE Est_Funds_Recovered_Clean IS NOT NULL
GROUP BY Agency
ORDER BY avg_recovered DESC
"""
sql_result = pd.read_sql(query, conn)
print("\nSQL query result — avg funds recovered per action, by agency:")
print(sql_result)

top_agency = sql_result.iloc[0]
print(f"\nAnswer: {top_agency['Agency']} has the highest average recovery "
      f"(${top_agency['avg_recovered']:,.0f} per action, {int(top_agency['num_actions'])} action(s)).")



--- 1.3 GOVERNMENT RESPONSE (SQL) ---

SQL query result — avg funds recovered per action, by agency:
                                              Agency  num_actions  avg_recovered  total_recovered
0  DOJ Consumer Protection Branch & U.S. Attorney...            2    74750000.00     1.495000e+08
1  DOJ Criminal Division's Money Laundering and A...            1    39600000.00     3.960000e+07
2  DOJ Civil Division, with U.S. Attorney's Offic...            1    15466278.00     1.546628e+07
3  U.S. Attorney's Office, District of New Mexico...            1    11000000.00     1.100000e+07
4           U.S. Attorney's Office, District of Utah            1     8400000.00     8.400000e+06
5  U.S. Attorney's Office, Northern District of N...            1     7168000.00     7.168000e+06
6  U.S. Attorney's Office, Middle District of Flo...            1     3244592.00     3.244592e+06
7  U.S. Attorney's Office, Western District of No...            1     1088554.99     1.088555e+06

Answer: DOJ Con

## Cross-Module Gap Analysis

In [8]:
print("\n--- CROSS-MODULE GAP ---")

top_scam_type_2024 = crime_types[crime_types['Year'] == 2024].sort_values(
    'Reported_Complaints', ascending=False).iloc[0]['Scam_Type']
top_scam_keywords = [w for w in re.split(r'[/\s]+', top_scam_type_2024) if len(w) > 3]
print(f"Most-reported scam type in 2024 (victim data): {top_scam_type_2024}")
print(f"Keywords used to search Government_Actions: {top_scam_keywords}")

pattern = '|'.join(top_scam_keywords)
matching_actions = gov_clean[gov_clean['Targeted_Scam_Type'].str.contains(
    pattern, case=False, na=False, regex=True)]
print(f"Number of the 10 logged gov actions specifically targeting '{top_scam_type_2024}': {len(matching_actions)}")
print("\nAll 10 targeted scam types logged in Government_Actions (for reference):")
print(gov_clean['Targeted_Scam_Type'].tolist())
print(f"\n--> GAP FINDING: '{top_scam_type_2024}' drives the single largest share of 2024 complaint "
      f"volume, yet only {len(matching_actions)} of the 10 logged enforcement actions target it directly. "
      f"Most actions instead cluster around Fraud/Medicare/Investment/Lottery cases.")



--- CROSS-MODULE GAP ---
Most-reported scam type in 2024 (victim data): Phishing/Spoofing
Keywords used to search Government_Actions: ['Phishing', 'Spoofing']
Number of the 10 logged gov actions specifically targeting 'Phishing/Spoofing': 0

All 10 targeted scam types logged in Government_Actions (for reference):
['Guardianship Fraud', 'Nursing Home / Medicare Fraud (False Claims Act)', 'Lottery/Sweepstakes Fraud', 'Data Broker Facilitation of Mass-Mailing Fraud', 'Romance Fraud', 'Fraud Committed by In-Home Service Providers', 'Investment Fraud (Boiler Room)', 'Nursing Home / Failure of Care (Civil False Claims Act)', 'Mail Fraud Scheme (Data Broker Facilitation)', 'Multiple Fraud Types (Grandparent, Lottery, Sweepstakes, Romance - via Western Union DPA)']

--> GAP FINDING: 'Phishing/Spoofing' drives the single largest share of 2024 complaint volume, yet only 0 of the 10 logged enforcement actions target it directly. Most actions instead cluster around Fraud/Medicare/Investment/Lotte

## Visualization


In [9]:
print("\n--- VISUALIZATION ---")

plt.figure(figsize=(9, 5.5))
sns.lineplot(data=age_trend, x='Year', y='Reported_Count', hue='Age_Group', marker='o', linewidth=2.5)
plt.title('IC3 Complaint Volume: Age 20-29 vs 60+ (2020-2024)')
plt.ylabel('Reported Complaints')
plt.tight_layout()
plt.savefig('age_trend.png', dpi=150)
plt.close()
print("age_trend.png")

plt.figure(figsize=(9, 6))
colors = plt.cm.OrRd([x / top_states_2024['Reported_Complaints'].max()
                       for x in top_states_2024.sort_values('Reported_Complaints')['Reported_Complaints']])
plt.barh(top_states_2024.sort_values('Reported_Complaints')['State'],
         top_states_2024.sort_values('Reported_Complaints')['Reported_Complaints'],
         color=colors)
plt.title('Top 10 States by Complaint Volume (2024) — Heat-Shaded')
plt.xlabel('Reported Complaints')
plt.tight_layout()
plt.savefig('top_states_2024.png', dpi=150)
plt.close()
print("top_states_2024.png")

plt.figure(figsize=(8, 5))
sns.barplot(data=top5_scam_types_2024, y='Scam_Type', x='Reported_Complaints', hue='Scam_Type', palette='mako', legend=False)
plt.title('Top 5 Scam Types by Complaint Volume (2024)')
plt.tight_layout()
plt.savefig('top_scam_types.png', dpi=150)
plt.close()
print("top_scam_types.png")

plt.figure(figsize=(8, 5))
sns.barplot(data=top3_payment_2024, y='Payment_Method', x='Reported_Count', hue='Payment_Method', palette='crest', legend=False)
plt.title('Top 3 Payment Methods Demanded (2024)')
plt.tight_layout()
plt.savefig('top_payment_methods.png', dpi=150)
plt.close()
print("top_payment_methods.png")

plt.figure(figsize=(8, 5))
sql_result['Agency_Short'] = sql_result['Agency'].str.slice(0, 40) + '...'
sns.barplot(data=sql_result, y='Agency_Short', x='avg_recovered', hue='Agency_Short', palette='flare', legend=False)
plt.title('Average Funds Recovered per Action, by Agency')
plt.xlabel('Avg Recovered (USD)')
plt.tight_layout()
plt.savefig('agency_recovery.png', dpi=150)
plt.close()
print("agency_recovery.png")



--- VISUALIZATION ---
age_trend.png
top_states_2024.png
top_scam_types.png
top_payment_methods.png
agency_recovery.png


##  Export Clean Outputs for Report

In [10]:
age_trend_pivot.to_csv('age_trend_data.csv')
top_states_2024.to_csv('top_states_2024_data.csv', index=False)
top5_scam_types_2024.to_csv('top_scam_types_data.csv', index=False)
top3_payment_2024.to_csv('top_payment_methods_data.csv', index=False)
sql_result.to_csv('agency_recovery_data.csv', index=False)
gov_clean.to_csv('government_actions_cleaned.csv', index=False)

print("\nAll analysis complete. CSVs and charts exported to /charts.")


All analysis complete. CSVs and charts exported to /charts.
